In [0]:
import requests
import json
from datetime import datetime, timezone

# Storage configuration
STORAGE_ACCOUNT = "marketpulsedatalake"
BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Stocks to ingest you can erase between runs the available values are: ["AAPL", "MSFT", "IBM"]
STOCKS = ["IBM"] 

# API key loaded securely from Azure Key Vault
API_KEY = dbutils.secrets.get(scope="market-pulse-secrets", key="alphavantage-api-key")

print("✅ Configuration loaded")
print(f"✅ Target stocks: {STOCKS}")
print(f"✅ Bronze path: {BRONZE_PATH}")

In [0]:
def ingest_stock(symbol: str, api_key: str) -> dict:
    """
    Calls the Alpha Vantage API and returns raw daily stock data.

    Args:
        symbol: Stock ticker (e.g. "IBM", "AAPL")
        api_key: Alpha Vantage API key

    Returns:
        dict with raw API response (Meta Data + Time Series)

    Raises:
        ValueError: If the API returns an error or rate limit is reached
    """
    url = "https://www.alphavantage.co/query"
    params = {
        "function": "TIME_SERIES_DAILY",  # daily historical data
        "symbol": symbol,
        "outputsize": "compact",           # last 100 trading days
        "apikey": api_key
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    # Check for API error message
    if "Error Message" in data:
        raise ValueError(f"API error for {symbol}: {data['Error Message']}")

    # Check for rate limit warning
    if "Note" in data:
        raise ValueError(f"API rate limit reached: {data['Note']}")

    print(f"✅ {symbol} — {len(data.get('Time Series (Daily)', {}))} days of data received")
    return data


def save_to_bronze(symbol: str, data: dict) -> str:
    """
    Saves raw JSON data to the ADLS bronze layer.
    Partitioned by ingestion date to support Auto Loader incremental reads.

    Bronze principle: append-only — existing files are never modified.
    Each run generates a new file with a unique timestamp.

    Args:
        symbol: Stock ticker (e.g. "IBM")
        data: Raw API response data

    Returns:
        Full path of the file created in ADLS
    """
    # Ingestion timestamp — timezone-aware UTC
    now = datetime.now(timezone.utc)
    ingest_ts = now.strftime("%Y%m%d_%H%M%S")
    ingest_date = now.strftime("%Y-%m-%d")

    # Path partitioned by date — enables efficient Auto Loader processing
    path = f"{BRONZE_PATH}/stocks/ingest_date={ingest_date}/{symbol}_{ingest_ts}.json"

    # Write to ADLS — each file is immutable once written
    dbutils.fs.put(path, json.dumps(data, indent=2), overwrite=True)

    print(f"✅ {symbol} saved to: {path}")
    return path


def ingest_all_stocks(stocks: list, api_key: str) -> list:
    """
    Orchestrates the ingestion of multiple stocks into the Bronze layer.
    Continues processing remaining stocks if one fails.

    Args:
        stocks: List of tickers to ingest (e.g. ["IBM", "AAPL"])
        api_key: Alpha Vantage API key

    Returns:
        List of file paths created in ADLS
    """
    paths = []
    errors = []

    for symbol in stocks:
        try:
            print(f"🔄 Ingesting {symbol}...")
            data = ingest_stock(symbol, api_key)
            path = save_to_bronze(symbol, data)
            paths.append(path)
        except Exception as e:
            # Log error but continue with remaining stocks
            print(f"❌ Error processing {symbol}: {e}")
            errors.append(symbol)

    # Ingestion summary
    print(f"\n✅ Ingestion complete: {len(paths)} stocks saved")
    if errors:
        print(f"❌ Failed stocks: {errors}")

    return paths


print("✅ Functions defined and ready")

In [0]:
# Run ingestion for all configured stocks
paths = ingest_all_stocks(STOCKS, API_KEY)

In [0]:
# Ver o que já existe no Bronze
files = dbutils.fs.ls("abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/")
for f in files:
    print(f.name, f"-", round(f.size/1024, 1), "KB")

In [0]:
# ATENTION!!! for project use only, please do not run is production!
# dbutils.fs.rm(
#     "abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-04-29/",
#     recurse=True
# )